<a href="https://colab.research.google.com/github/MightyCrimsonX/Crimson-Notebooks/blob/main/MightyCrimson_SimpleAnima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 📥 Celda 1: Instalar ComfyUI Backend y Descargar Componentes de Anima
#@markdown ---

import os as _os
import subprocess as _sp
import time as _time
import socket as _socket
from IPython.display import clear_output

print("============================================================")
print("📦 CONFIGURANDO ENTORNO COMFYUI CON LIVE PREVIEWS...")
print("============================================================")
%cd /content
# 1. Clonar ComfyUI si no existe
if not _os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd ComfyUI

# 2. Instalar dependencias esenciales
print("\n⚡ Instalando librerías del sistema...")
!uv pip install -U transformers accelerate safetensors sentencepiece protobuf
!uv pip install torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 xformers==0.0.33.post2 triton==3.5.1 --index-url https://download.pytorch.org/whl/cu128 --no-progress
!uv pip install "numpy==2.0.2" websocket-client tqdm
!uv pip install -r requirements.txt
!pip install aria2

# 3. Asegurar rutas de almacenamiento y descargar componentes de Anima (Split Files)
_os.makedirs("models/diffusion_models", exist_ok=True)
_os.makedirs("models/text_encoders", exist_ok=True)
_os.makedirs("models/vae", exist_ok=True)
_os.makedirs("models/upscale_models", exist_ok=True)

BASE_ANIMA_URL = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files"
if not _os.path.exists("models/diffusion_models/anima-base-v1.0.safetensors"):
    print("   ↳ Descargando Transformer de Anima...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{BASE_ANIMA_URL}/diffusion_models/anima-base-v1.0.safetensors" -d models/diffusion_models -o anima-base-v1.0.safetensors
if not _os.path.exists("models/text_encoders/qwen_3_06b_base.safetensors"):
    print("   ↳ Descargando Text Encoder (Qwen)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{BASE_ANIMA_URL}/text_encoders/qwen_3_06b_base.safetensors" -d models/text_encoders -o qwen_3_06b_base.safetensors
if not _os.path.exists("models/vae/qwen_image_vae.safetensors"):
    print("   ↳ Descargando VAE Dedicado...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{BASE_ANIMA_URL}/vae/qwen_image_vae.safetensors" -d models/vae -o qwen_image_vae.safetensors

# Descargar upscalers solicitados
print("\n🔄 Verificando modelos de Upscale...")
upscalers = {
    "remacri_original.pt": "https://huggingface.co/LyliaEngine/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.safetensors",
    "4x-AnimeSharp.pth": "https://huggingface.co/Kim2091/AnimeSharp/resolve/main/4x-AnimeSharp.pth",
    "RealESRGAN_x4plus_anime_6B.pth": "https://huggingface.co/gemasai/RealESRGAN_x4plus_anime_6B/resolve/main/RealESRGAN_x4plus_anime_6B.pth",
    "4x_IllustrationJaNai_V1_ESRGAN_135k.pth": "https://huggingface.co/halllooo/4x_illustrationJaNaiV1/resolve/main/4x_IllustrationJaNai_V1_ESRGAN_135k.pth",
    "2xLexicaSwinIR.pth": "https://github.com/Phhofm/models/raw/main/2xLexicaSwinIR/2xLexicaSwinIR.pth",
    "4xmssim_drct-l_pretrain.pth": "https://github.com/Phhofm/models/releases/download/4xDRCT-mssim-pretrains/4xmssim_drct-l_pretrain.pth"
}
for name, url in upscalers.items():
    if not _os.path.exists(f"models/upscale_models/{name}"):
        print(f"   ↳ Adquiriendo {name}...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d models/upscale_models -o "{name}"

# 4. Lanzar ComfyUI Headless con redirección de logs
print("\n🚀 Lanzando backend de ComfyUI...")
!fuser -k 8188/tcp > /dev/null 2>&1

log_file = open("comfyui.log", "w")
comfy_process = _sp.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--preview-method", "latent2rgb"
], stdout=log_file, stderr=log_file)
clear_output()
# 5. Verificación de estabilidad y puerto
print("⏳ Esperando respuesta del servidor web (Puerto 8188)...")
server_ready = False
for i in range(60):
    _time.sleep(1)

    # Comprobar si el proceso murió prematuramente
    if comfy_process.poll() is not None:
        print("\n❌ ¡El servidor de ComfyUI se estrelló durante el arranque!")
        break

    s = _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM)
    try:
        s.connect(("127.0.0.1", 8188))
        s.close()
        server_ready = True
        print("\n============================================================")
        print("✅ ¡Todo Listo!")
        print("============================================================")
        break
    except _socket.error:
        print(".", end="")

if not server_ready:
    print("\n============================================================")
    print("📋 [DIAGNÓSTICO DE CRASH DE COMFYUI]:")
    print("============================================================")
    with open("comfyui.log", "r") as f:
        print(f.read())
print("============================================================")

In [ ]:
#@title 📥 Celda 2: Iniciar ComfyUI Backend (Por si apagaste la celda de generacion antes de tiempo)
#@markdown ---
import os as _os
import subprocess as _sp
import time as _time
import socket as _socket
from IPython.display import clear_output
# 4. Lanzar ComfyUI Headless con redirección de logs
print("\n🚀 Lanzando backend de ComfyUI...")
!fuser -k 8188/tcp > /dev/null 2>&1

log_file = open("comfyui.log", "w")
comfy_process = _sp.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--preview-method", "latent2rgb"
], stdout=log_file, stderr=log_file)
clear_output()
# 5. Verificación de estabilidad y puerto
print("⏳ Esperando respuesta del servidor web (Puerto 8188)...")
server_ready = False
for i in range(60):
    _time.sleep(1)

    # Comprobar si el proceso murió prematuramente
    if comfy_process.poll() is not None:
        print("\n❌ ¡El servidor de ComfyUI se estrelló durante el arranque!")
        break

    s = _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM)
    try:
        s.connect(("127.0.0.1", 8188))
        s.close()
        server_ready = True
        print("\n============================================================")
        print("✅ ¡Todo Listo!")
        print("============================================================")
        break
    except _socket.error:
        print(".", end="")

if not server_ready:
    print("\n============================================================")
    print("📋 [DIAGNÓSTICO DE CRASH DE COMFYUI]:")
    print("============================================================")
    with open("comfyui.log", "r") as f:
        print(f.read())
print("============================================================")

In [ ]:
#@title 🖼️ Celda 3: Configurar Parámetros, Upscale y Generar
#@markdown ### La primera vez al generar se demora al cargar los modelos una vez. No se preocupen si ven un "none" cuando generan.
#@markdown ---
#@markdown ### Prompts
positive_prompt = "masterpiece, best quality, 1girl," #@param {type:"string"}
negative_prompt = "worst quality, low quality, signature, watermark, username, blurry, deformed" #@param {type:"string"}

#@markdown ---
#@markdown ### Ajustes de Upscaling (Súper Resolución)
activar_upscale = False #@param {type:"boolean"}
modelo_upscale = "4x-AnimeSharp.pth" #@param ["remacri_original.pt", "4x-AnimeSharp.pth", "RealESRGAN_x4plus_anime_6B.pth", "4x_IllustrationJaNai_V1_ESRGAN_135k.pth", "2xLexicaSwinIR.pth", "4xmssim_drct-l_pretrain.pth"]

#@markdown ---
#@markdown ### Configuración del KSampler
cfg_scale = 4.5 #@param {type:"slider", min:1.0, max:20.0, step:0.5}
steps = 25 #@param {type:"slider", min:10, max:100, step:1}
resolution = "832x1216" #@param ["832x1216", "768x1334", "896x1152", "1024x1024","1334x768", "1216x832", "1216x832", "1152x896"]
sampler = "euler_ancestral" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde_gpu"]
scheduler = "sgm_uniform" #@param ["normal", "karras", "exponential", "sgm_uniform"]
seed = -1 #@param {type:"integer"}

import websocket
import uuid
import json
import urllib.request
import urllib.parse
import urllib.error
import torch
from PIL import Image
import io
from IPython.display import display, clear_output
from IPython.display import DisplayHandle
from tqdm import tqdm

server_address = "127.0.0.1:8188"
client_id = str(uuid.uuid4())

width, height = map(int, resolution.replace(",", "x").split("x"))
if seed < 0:
    seed = torch.randint(0, 2**32 - 1, (1,)).item()

# Tu mapa de nodos base original (Garantizado que funciona)
workflow_api = {
  "1": { "inputs": { "unet_name": "anima-base-v1.0.safetensors", "weight_dtype": "default" }, "class_type": "UNETLoader" },
  "2": { "inputs": { "clip_name": "qwen_3_06b_base.safetensors", "type": "stable_diffusion" }, "class_type": "CLIPLoader" },
  "3": { "inputs": { "vae_name": "qwen_image_vae.safetensors" }, "class_type": "VAELoader" },
  "4": { "inputs": { "text": positive_prompt, "clip": ["2", 0] }, "class_type": "CLIPTextEncode" },
  "5": { "inputs": { "text": negative_prompt, "clip": ["2", 0] }, "class_type": "CLIPTextEncode" },
  "46": { "inputs": { "width": width, "height": height, "batch_size": 1 }, "class_type": "EmptyLatentImage" },
  "7": {
    "inputs": {
      "seed": seed, "steps": steps, "cfg": cfg_scale, "sampler_name": sampler, "scheduler": scheduler,
      "denoise": 1.0, "model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["46", 0]
    },
    "class_type": "KSampler"
  },
  "8": { "inputs": { "samples": ["7", 0], "vae": ["3", 0] }, "class_type": "VAEDecode" }
}

# Inyección dinámica con el parámetro corregido a 'model_name'
if activar_upscale:
    workflow_api["10"] = {
        "inputs": { "model_name": modelo_upscale }, # <-- FIJADO: Nombre de parámetro correcto para la API
        "class_type": "UpscaleModelLoader"
    }
    workflow_api["11"] = {
        "inputs": { "upscale_model": ["10", 0], "image": ["8", 0] },
        "class_type": "ImageUpscaleWithModel"
    }
    workflow_api["9"] = {
        "inputs": { "filename_prefix": "Anima_Upscaled", "images": ["11", 0] },
        "class_type": "SaveImage"
    }
else:
    workflow_api["9"] = {
        "inputs": { "filename_prefix": "Anima_Live_Preview", "images": ["8", 0] },
        "class_type": "SaveImage"
    }

def queue_prompt(prompt_workflow):
    p = {"prompt": prompt_workflow, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    req = urllib.request.Request(f"http://{server_address}/prompt", data=data)
    try:
        return json.loads(urllib.request.urlopen(req).read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        error_body = e.read().decode('utf-8')
        print("\n❌ [ComfyUI Server Validation Error]:")
        try:
            print(json.dumps(json.loads(error_body), indent=2))
        except:
            print(error_body)
        raise RuntimeError("ComfyUI rechazó la estructura de nodos.")

def get_image(filename, subfolder, folder_type):
    data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
    url_values = urllib.parse.urlencode(data)
    with urllib.request.urlopen(f"http://{server_address}/view?{url_values}") as response:
        return response.read()

def generate_image_with_preview(prompt_workflow):
    ws = websocket.WebSocket()
    ws.connect(f"ws://{server_address}/ws?clientId={client_id}")

    print("🎨 Enviando flujo a ComfyUI...")
    prompt_id = queue_prompt(prompt_workflow)['prompt_id']

    pbar = None
    preview_handle = DisplayHandle()
    preview_handle.display(print("⏳ Esperando primer paso de renderizado..."))

    while True:
        out = ws.recv()
        if isinstance(out, str):
            message = json.loads(out)
            if message['type'] == 'progress':
                data = message['data']
                if pbar is None:
                    pbar = tqdm(
                        total=data['max'],
                        desc="📥 Generando pasos",
                        bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
                    )
                pbar.n = data['value']
                pbar.refresh()

            if message['type'] == 'executing':
                data = message['data']
                if data['node'] is None and data['prompt_id'] == prompt_id:
                    break
        else:
            try:
                image_bytes = out[8:]
                preview_img = Image.open(io.BytesIO(image_bytes))
                preview_handle.update(preview_img)
            except Exception:
                pass

    if pbar is not None:
        pbar.close()

    if activar_upscale:
        print(f"🚀 Aplicando súper resolución con {modelo_upscale}...")
    else:
        print("✨ Decodificando alta resolución final con el VAE...")

    with urllib.request.urlopen(f"http://{server_address}/history/{prompt_id}") as response:
        history = json.loads(response.read().decode('utf-8'))[prompt_id]

    for node_id in history['outputs']:
        node_output = history['outputs'][node_id]
        if 'images' in node_output:
            for image_info in node_output['images']:
                image_data = get_image(image_info['filename'], image_info['subfolder'], image_info['type'])
                ws.close()
                return Image.open(io.BytesIO(image_data))
    ws.close()
    raise RuntimeError("Error al recuperar la imagen.")

# Ejecución del bloque
try:
    output_image = generate_image_with_preview(workflow_api)
    clear_output(wait=True)
    status_msg = f"✅ ¡Inferencia completada con Upscale ({modelo_upscale})!" if activar_upscale else "✅ ¡Inferencia completada exitosamente!"
    print(status_msg)
    print(f"📌 Semilla: {seed} | Dimensión final: {output_image.width}x{output_image.height}")

    output_image.save(f"anima_preview_out_{seed}.png")
    display(output_image)
except Exception as e:
    print(f"\n❌ Detalle del fallo: {e}")